# 牛奶产量 ARIMA 预测

使用 1962-1975 年月度牛奶产量数据，保留最后 6 个月作为测试集，训练 ARIMA 模型并评估预测误差。


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print


RANDOM_STATE = 42


def find_project_root(start=None):
    """Find the project root by walking upward to the data directory.

    Args:
        start: Optional start path. Defaults to the current working directory.

    Returns:
        Project root path containing the `data` directory.

    Raises:
        FileNotFoundError: If no parent directory contains `data`.
    """
    current = Path.cwd() if start is None else Path(start).resolve()
    for path in [current, *current.parents]:
        if (path / "data").exists():
            return path
    raise FileNotFoundError("未找到包含 data/ 的项目根目录")


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"

plt.rcParams["font.sans-serif"] = ["SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


from itertools import product

from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller


## 1. 读取数据并划分训练/测试集


In [ ]:
milk = pd.read_csv(DATA_DIR / "milk_production.csv", parse_dates=["Month"])
series = milk.set_index("Month")["production"].asfreq("MS")
train = series.iloc[:-6]
test = series.iloc[-6:]

fig, ax = plt.subplots(figsize=(10, 4), dpi=120)
train.plot(ax=ax, label="训练集")
test.plot(ax=ax, label="测试集")
ax.set_title("月度牛奶产量")
ax.set_ylabel("产量")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

display(pd.concat({"train_tail": train.tail(6), "test": test}, axis=1))


## 2. 分解与平稳性检验


In [ ]:
decomposition = seasonal_decompose(train, model="additive", period=12)
decomposition.plot()
plt.show()

def stationarity_report(values, name):
    """Build ADF and Ljung-Box diagnostics for a time series.

    Args:
        values: Time series values to diagnose.
        name: Display name used in the report.

    Returns:
        Dictionary containing stationarity and white-noise test statistics.
    """
    adf_stat, p_value, used_lag, n_obs, critical_values, _ = adfuller(
        values.dropna(), autolag="AIC"
    )
    ljung_box = acorr_ljungbox(values.dropna(), lags=[12], return_df=True)
    return {
        "series": name,
        "adf_stat": adf_stat,
        "adf_p_value": p_value,
        "used_lag": used_lag,
        "n_obs": n_obs,
        "critical_1%": critical_values["1%"],
        "critical_5%": critical_values["5%"],
        "ljung_box_p_value_lag12": ljung_box["lb_pvalue"].iloc[0],
    }


reports = [
    stationarity_report(train, "原序列"),
    stationarity_report(train.diff().dropna(), "一阶差分"),
    stationarity_report(train.diff().diff().dropna(), "二阶差分"),
]
display(pd.DataFrame(reports).round(4))


## 3. ACF 与 PACF


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), dpi=120)
plot_acf(train.diff().dropna(), lags=36, ax=axes[0])
plot_pacf(train.diff().dropna(), lags=36, method="ywm", ax=axes[1])
axes[0].set_title("一阶差分 ACF")
axes[1].set_title("一阶差分 PACF")
plt.tight_layout()
plt.show()


## 4. 使用 BIC 搜索 ARIMA 参数


In [ ]:
def choose_d(values, max_d=2, alpha=0.05):
    """Choose a differencing order using the ADF p-value.

    Args:
        values: Training time series.
        max_d: Maximum differencing order to test.
        alpha: Significance level for the ADF test.

    Returns:
        First differencing order whose ADF p-value is below `alpha`.
    """
    def difference(data, order):
        """Apply ordinary differencing repeatedly.

        Args:
            data: Source time series.
            order: Number of differencing rounds.

        Returns:
            Differenced series with missing values removed.
        """
        result = data.copy()
        for _ in range(order):
            result = result.diff().dropna()
        return result

    for d in range(max_d + 1):
        candidate = difference(values, d) if d else values.dropna()
        p_value = adfuller(candidate, autolag="AIC")[1]
        if p_value < alpha:
            return d
    return max_d


def search_arima_order(values, d, max_p=5, max_q=5):
    """Search ARIMA orders and rank them by BIC.

    Args:
        values: Training time series.
        d: Differencing order.
        max_p: Maximum autoregressive order to test.
        max_q: Maximum moving-average order to test.

    Returns:
        Data frame sorted by BIC with candidate ARIMA orders.
    """
    records = []
    for p, q in product(range(max_p + 1), range(max_q + 1)):
        if p == 0 and d == 0 and q == 0:
            continue
        try:
            result = ARIMA(
                values,
                order=(p, d, q),
                enforce_stationarity=False,
                enforce_invertibility=False,
            ).fit()
        except Exception:
            continue
        records.append({"p": p, "d": d, "q": q, "aic": result.aic, "bic": result.bic})
    return pd.DataFrame(records).sort_values("bic").reset_index(drop=True)


d = choose_d(train)
order_report = search_arima_order(train, d=d, max_p=5, max_q=5)
best_order = tuple(order_report.loc[0, ["p", "d", "q"]].astype(int))

print(f"推荐 ARIMA 参数: {best_order}")
display(order_report.head(10).round(2))


## 5. 训练模型并预测


In [ ]:
model = ARIMA(
    train,
    order=best_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit()

forecast = model.forecast(steps=len(test))
forecast.index = test.index

prediction = pd.DataFrame({"actual": test, "forecast": forecast})
prediction["error"] = prediction["actual"] - prediction["forecast"]

mae = mean_absolute_error(prediction["actual"], prediction["forecast"])
rmse = np.sqrt(mean_squared_error(prediction["actual"], prediction["forecast"]))
mape = (prediction["error"].abs() / prediction["actual"]).mean()

fig, ax = plt.subplots(figsize=(10, 4), dpi=120)
series.plot(ax=ax, label="实际值")
forecast.plot(ax=ax, label="预测值")
ax.axvline(test.index[0], color="gray", linestyle="--", alpha=0.8)
ax.set_title("ARIMA 牛奶产量预测")
ax.set_ylabel("产量")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

display(prediction.round(2))
display(pd.DataFrame({"MAE": [mae], "RMSE": [rmse], "MAPE": [mape]}).round(4))
print(model.summary())
